# Project Drive setup
Mount Drive in Colab, verify a write and read, and prepare results_archive/run_logs for durable experimental logs.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
import json
import shutil
from uuid import uuid4

root = Path("/content/drive/MyDrive/Research/Criticality_Evolution")
for directory in ("datasets/mouse", "datasets/zebra_finch", "processed", "cache",
                  "checkpoints", "matlab_inputs", "results_archive", "results_archive/run_logs", "manifests"):
    (root / directory).mkdir(parents=True, exist_ok=True)

probe_id = uuid4().hex
target = root / "manifests" / f"drive-probe-{probe_id}.json"
staged = target.with_suffix(".tmp")
payload = json.dumps({
    "probe_id": probe_id,
    "written_at_utc": datetime.now(timezone.utc).isoformat(),
    "purpose": "Colab Drive persistence verification",
}).encode()
staged.write_bytes(payload)
staged.replace(target)
readback = target.read_bytes()
assert readback == payload

usage = shutil.disk_usage(root)
print(json.dumps({
    "status": "PASS",
    "relative_path": str(target.relative_to(root)),
    "bytes": len(readback),
    "sha256": sha256(readback).hexdigest(),
    "drive_total_bytes": usage.total,
    "drive_free_bytes": usage.free,
}, indent=2))